<a href="https://colab.research.google.com/github/devendra4014/LLM-Lab/blob/main/notebooks/gpt2-dev.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The LLM Architecture Blueprint: From Math to Code

This document breaks down the forward pass of a standard Autoregressive Transformer (like GPT).

**Our Baseline Configuration:**
* **Sequence Length (`T` or `block_size`):** 1024 tokens
* **Embedding Dimension (`C` or `n_embd`):** 384
* **Number of Heads (`n_heads`):** 6
* **Head Dimension (`head_dim`):** 64 (because $384 / 6 = 64$)
* **Batch Size (`B`):** 1 (kept at 1 for simpler math)


---

## 1. Input Preparation: Embeddings & Positional Encoding

**What it is:** The process of converting raw text into numbers, and then adding spatial awareness to those numbers.
* **Token Embedding:** Maps an integer (token ID) to a dense vector of size 384.
* **Positional Encoding:** Adds a unique pattern of numbers to each token's vector based on its position (0 to 1023).

**Why we need it:** Unlike older models (RNNs) that read text sequentially word-by-word, Transformers read the entire 1024-token chunk simultaneously. Without positional encoding, the model would see "The dog bit the man" and "The man bit the dog" as the exact same bag of words.

**Advantages:** Enables massive parallel processing on GPUs since we don't have to wait for token $t$ to process token $t+1$.

**The Data Flow & Shapes:**
1.  **Raw Input:** `(1024)` -> 1D sequence of integers.
2.  **Token Embedding:** `(1024, 384)`
3.  **+ Positional Encoding:** `(1024, 384)` -> *This is the input $x$ that enters the Transformer Block.*

---

## 2. The Core Engine: Self-Attention

**What it is:** The mechanism that allows each token to look at every other token in the sequence to gather context.



**Why we need it:** Words change meaning based on context. In the sentence "The bank of the river," "bank" means land. In "The bank on the corner," it means a financial institution. Self-attention allows the vector for "bank" to pull information from "river" or "corner" to update its own meaning.

**Advantages:** Creates a "global receptive field." A token at position 1000 can directly look at a token at position 1 in a single computational step.

**The Data Flow & Shapes:**
1.  **Create Q, K, V:** We multiply input $x$ by three learned weight matrices ($W_q, W_k, W_v$).
    * **Shape:** `(1024, 384)` each.
    * *Code Map:* `qkv = self.c_attn(x)` then `q, k, v = qkv.split(...)`
2.  **Raw Scores:** $Q \times K^T$
    * **Shape:** `(1024, 1024)` -> *An attention grid showing how every token relates to every other token.*
3.  **Scale:** Divide by $\sqrt{d_k}$ (which is 8 in our case).
    * **Why?** If dot products get too large, the subsequent Softmax function gets pushed into flat regions where gradients vanish (the model stops learning). Scaling keeps the variance stable.
4.  **Softmax:** Converts the grid into percentages that sum to 1.0 across rows.
5.  **Context Matrix:** Multiply probabilities by $V$.
    * **Shape:** `(1024, 384)`

---

## 3. Parallel Processing: Multi-Head Attention (MHA)

**What it is:** Instead of calculating one large attention grid using the full 384 dimensions, we slice the embeddings into 6 smaller chunks (heads) of 64 dimensions each.



**Why we need it:** A single word plays multiple roles in a sentence. It has a grammatical role (noun/verb), an emotional tone (positive/negative), and a semantic link (who it refers to). Splitting into "heads" allows the model to learn multiple different relationships simultaneously.

**Advantages:** Massively improves the richness of the representations. Head 1 might learn to track pronouns, while Head 2 learns to track punctuation.

**The Data Flow & Shapes:**
1.  **Reshape & Transpose:** Slice the 384 dims into 6 heads of 64.
    * **Shape:** `(1, 6, 1024, 64)`
    * *Code Map:* `q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)`
2.  **Parallel Attention:** The $Q \times K^T$ math happens 6 times at once.
    * **Shape:** `(1, 6, 1024, 1024)` -> *We now have six separate attention grids!*
3.  **Apply Values:** Multiply by $V$.
    * **Shape:** `(1, 6, 1024, 64)`
4.  **Recombine:** Concatenate the 6 heads back together.
    * **Shape:** `(1, 1024, 384)`
    * *Code Map:* `y.transpose(1, 2).contiguous().view(B, T, C)`

---

## 4. Autoregressive Enforcement: Causal Masking

**What it is:** A lower-triangular matrix of $1$s and $0$s used to overwrite future attention scores with $-\infty$.



**Why we need it:** During training, we feed the model the entire 1024-word sequence at once for GPU efficiency. However, the model's job is to predict the *next* word. If token #5 can look at token #6 in the attention grid, it's cheating.

**Advantages:** Allows us to train on thousands of words in parallel while strictly enforcing the rule that tokens can only gather context from the past and the present.

**The Data Flow & Shapes:**
1.  **Mask Application:** Overlay the mask on the raw `(1024, 1024)` attention grid (per head).
    * *Code Map:* `attn_scores.masked_fill(self.mask[:,:, :T, :T] == 0, float('-inf'))`
2.  **Softmax Effect:** $e^{-\infty} = 0$. The probabilities of looking at any future token become exactly 0%.

---

## 5. The Reasoning Engine: Feed-Forward Network (FFN)

**What it is:** A standard multi-layer perceptron (MLP) applied to each token's 384-dimensional vector individually.

**Why we need it:** Attention just moves information *between* tokens. The FFN is where the model actually "thinks" about what that new information means. It acts as a memory bank and a pattern recognizer.

**Advantages:** The non-linear activation function (like GELU) allows the model to map complex, non-linear representations of language that simple matrix multiplication cannot capture.

**The Data Flow & Shapes:**
1.  **Expansion Layer:** Usually scales up by 4x.
    * **Shape:** `(1024, 1536)`
2.  **Activation:** Apply GELU.
3.  **Contraction Layer:** Project back down.
    * **Shape:** `(1024, 384)`

---

## 6. Optimization: Standard Math vs. Flash Attention

In your PyTorch code, you implemented a switch between `flash-attention` and `own` (standard math). Here is why that switch matters:

**The Problem with Standard Math:**
Creating that `(1, 6, 1024, 1024)` attention grid takes up a massive amount of VRAM. If your `block_size` increases from 1024 to 8000, the grid size grows quadratically ($O(N^2)$), causing Out-Of-Memory (OOM) errors. Furthermore, moving this massive grid between the GPU's main memory (HBM) and its processing cores (SRAM) is incredibly slow.

**The Flash Attention Solution:**
Flash Attention is a hardware-aware algorithm. Instead of calculating the whole `1024x1024` grid and saving it to main memory, it computes the attention in small "blocks" directly on the ultra-fast SRAM chip, applies the softmax block-by-block, and only saves the final `(1024, 64)` output back to main memory.

* **Advantage:** It mathematically yields the *exact same result* as standard attention, but uses a fraction of the memory and is significantly faster.
* *Code Map:* `F.scaled_dot_product_attention(..., is_causal=True)` automatically triggers the Flash Attention backend in PyTorch 2.0+.

***


In [1]:
import kagglehub
path = kagglehub.dataset_download("tomasbebra/tiny-stories-ds")

Using Colab cache for faster access to the 'tiny-stories-ds' dataset.


In [3]:
from dataclasses import dataclass

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR

torch.backends.cudnn.benchmark = True

## GELU (Gaussian Error Linear Unit)
The **GELU activation function** is defined as:

$$
\text{GELU}(x) = x \cdot \Phi(x)
$$

Where:

* ($\Phi(x)$) is the **cumulative distribution function (CDF)** of the standard normal distribution.




### Approximation (Tanh-based):

A commonly used fast approximation is:

$$
\text{GELU}(x) \approx 0.5x \left(1 + \tanh\left(\sqrt{\frac{2}{\pi}} \left(x + 0.044715 x^3 \right)\right)\right)
$$

---


* GELU can be seen as **stochastic regularization**:

  * Instead of hard gating like ReLU, it **weights inputs smoothly**.
* It keeps small negative values (unlike ReLU which zeros them).
* Acts like a **smooth version of ReLU**.

---

### Properties

* **Smooth and differentiable everywhere**
* Non-monotonic (slight curvature in negative region)
* Better gradient flow than ReLU
* Works well in deep architectures



In [ ]:
class AppGELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        out = 0.5 * x * (1 + torch.tanh(torch.sqrt(torch.tensor(2.0/torch.pi)) * (x + 0.044715 * torch.pow(x, 3))))
        return out

In [ ]:
import matplotlib.pyplot as plt

relu, gelu = nn.ReLU(), AppGELU()

x = torch.linspace(-3, 3, 100)
y_gelu, y_relu = gelu(x), relu(x)

plt.figure(figsize=(8, 3))

for i, (y, lable) in enumerate(zip([y_gelu, y_relu], ['GELU', 'RELU']), 1):
    plt.subplot(1, 2, i)
    plt.plot(x, y)
    plt.title(f"{lable} activation function")
    plt.xlabel("x")
    plt.ylabel(f"{lable}(x)")
    plt.grid(True)

plt.tight_layout()
plt.show()

## dataset class

In [2]:

class TextDataset(Dataset):
    def __init__(self, txt, tokeniser, block_size):
        self.block_size = block_size
        self.tokens = torch.tensor(tokeniser.encode(txt))
    
    def __len__(self):
        return len(self.tokens) - self.block_size

    def __getitem__(self, idx):
        x = self.tokens[idx: idx + self.block_size]
        y = self.tokens[idx + 1: idx + self.block_size + 1]
        return x, y

def get_data_loader(dataset, batch_size):
    import os
    num_workers = 0 # min(4, os.cpu_count() or 2)

    data_loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
        persistent_workers=True if num_workers > 0 else False
    )
    return data_loader



## GPT Implementation

In [4]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        # Ensure embedding dimension is perfectly divisible by the number of heads
        assert config.n_embd % config.n_head == 0

        self.n_heads = config.n_head
        self.n_embd = config.n_embd
        self.head_dim = config.n_embd // config.n_head

        # Read attention mode from config (defaulting to flash-attention)
        self.attention_mode = getattr(config, 'attention_type', 'flash-attention')
        self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention')

        # Q, K, V projections bundled into one linear layer for speed
        self.c_attn = nn.Linear(config.n_embd, config.n_embd * 3, bias=config.bias)
        # Output projection
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)

        # Regularization
        self.attn_dropout = config.attn_pdrop
        self.resid_dropout = config.resid_pdrop

        # Causal mask (only necessary for the 'own' implementation)
        # We use register_buffer so PyTorch moves it to the correct device automatically
        if self.attention_mode == 'manual' or not self.flash:
            # Create causal mask for non-flash attention
            self.register_buffer(
                "mask",
                torch.tril(torch.ones(config.block_size, config.block_size))
                .view(1, 1, config.block_size, config.block_size)
            )

    def forward(self, x):
        B, T, C = x.size() # Batch size, Sequence length, Embedding dimension

        # 1. Project to Q, K, V
        qkv = self.c_attn(x) # (B, T, n_embd * 3)
        q, k, v = qkv.split(self.n_embd, dim=2) # Split into three (B, T, n_embd) tensors

        # 2. Reshape and transpose for multi-head attention
        # Shape becomes: (B, n_heads, T, head_dim)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        # 3. Handle dropout train/inference state for functional attention
        current_dropout_p = self.attn_dropout if self.training else 0.0

        # 4. Compute Attention based on the selected mode
        if self.attention_mode == 'flash-attention' and self.flash:
            # PyTorch 2.0+ handles FlashAttention natively if is_causal=True and no mask is passed.
            y = F.scaled_dot_product_attention(
                q, k, v,
                attn_mask=None,
                dropout_p=current_dropout_p,
                is_causal=True
            )

        elif self.attention_mode == 'without-flash-attention':
            # We explicitly force PyTorch's SDPA to disable the flash backend.
            # It will fall back to memory-efficient or standard math attention.
            with torch.backends.cuda.sdp_kernel(enable_flash=False, enable_math=True, enable_mem_efficient=True):
                y = F.scaled_dot_product_attention(
                    q, k, v,
                    attn_mask=None,
                    dropout_p=current_dropout_p,
                    is_causal=True
                )

        else:
            # 'own' implementation: Manual Math
            # Q @ K^T / sqrt(d)
            attn_scores = (q @ k.transpose(-2, -1)) * (1.0 / (self.head_dim ** 0.5))

            # Apply causal mask (block out future tokens)
            attn_scores = attn_scores.masked_fill(self.mask[:, :, :T, :T] == 0, float('-inf'))

            # Softmax to get probabilities
            attn_scores = F.softmax(attn_scores, dim=-1)

            # Apply attention dropout (using the nn.Module so it respects model.eval())
            attn_scores = F.dropout(attn_scores, p=self.attn_dropout, training=self.training)

            # Multiply by Values
            y = attn_scores @ v # (B, n_heads, T, head_dim)

        # 5. Re-assemble multi-head outputs and apply final projection
        y = y.transpose(1, 2).contiguous().view(B, T, C)

        # Residual dropout and output projection
        y = self.c_proj(y)
        y = F.dropout(y, p=self.resid_dropout, training=self.training)

        return y


class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.c_fc = nn.Linear(config.n_embd , 4 * config.n_embd, bias=config.bias)
        self.gelu = nn.GELU(approximate="tanh")
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)

        return x


class Block(nn.Module):
    """ Transformer Block with self-attention and MLP Layer followed by Layer Normalization with residual connection. """
    def __init__(self, config):
        super().__init__()

        self.ln_1 = nn.LayerNorm(config.n_embd, bias=config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd, bias=config.bias)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


class GPT(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.config = config

        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd), # token embedding
            wpe = nn.Embedding(config.block_size, config.n_embd), # positional encoding
            emb_drop = nn.Dropout(config.embd_pdrop),
            h = nn.Sequential(*[Block(config) for _ in range(config.n_layer)]), # attention blocks
            ln_f = nn.LayerNorm(config.n_embd)
        ))

        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.lm_head.weight = self.transformer.wte.weight

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
        elif isinstance(module, nn.LayerNorm):
            torch.nn.init.zeros_(module.bias)
            torch.nn.init.ones_(module.weight)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        assert T <= self.config.block_size , f"input sequence of length `{T}` is greater than context length of `{self.config.block_size}`."

        tok_embeds = self.transformer.wte(idx)
        pos_embeds = self.transformer.wpe(torch.arange(T, device=idx.device).unsqueeze(0))
        x = tok_embeds + pos_embeds  # Shape [batch_size, num_tokens, emb_size]
        x = self.transformer.emb_drop(x)
        x = self.transformer.h(x)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))

        return logits, loss

    @classmethod
    def from_pretrained(cls, model_type='gpt2'):
        from transformers import GPT2LMHeadModel

        # get the GPT-2 124M parameter model from huggingface transformer library
        download_dir = r"E:\code\DL\LLM-Lab\.model"
        model_hf = GPT2LMHeadModel.from_pretrained('gpt2', cache_dir=download_dir)
        sd_hf = model_hf.state_dict()
        sd_keys_hf = sd_hf.keys()

        # get the config for GPT-2 model
        config = GPTConfig

        # set the required parameters for GPT model
        config.vocab_size = 50257
        config.block_size = 1024
        config.n_layer = 12
        config.n_head = 12

        # create the model
        model = GPT(config)
        sd = model.state_dict()
        sd_keys = sd.keys()

        assert len(sd_keys_hf) == len(sd_keys), f"mismatched keys: {len(sd_keys_hf)} != {len(sd_keys)}"

        transposed = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']
        for k in sd_keys_hf:
            if any(k.endswith(w) for w in transposed):
                # special treatment for the Conv1D weights we need to transpose
                assert sd_hf[k].shape[::-1] == sd[k].shape, f"{sd_hf[k].shape[::-1]}, and {sd[k].shape}"
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k].t())
            else:
                # vanilla copy over the other parameters
                assert sd_hf[k].shape == sd[k].shape, f"{sd_hf[k].shape} and {sd[k].shape}"
                with torch.no_grad():
                    sd[k].copy_(sd_hf[k])

        return model

    def count_parameters(self, trainable_only: bool = False) -> int:
        """
        Count the number of parameters in this model.
        Args:
            trainable_only (bool): If True, counts only parameters with requires_grad=True
        Returns:
            int: Total number of parameters
        """
        if trainable_only:
            return sum(p.numel() for p in self.parameters() if p.requires_grad)
        else:
            return sum(p.numel() for p in self.parameters())

## Text generation method

In [4]:
@torch.no_grad()
def generate(model, idx, max_tokens, temperature=1.0, top_k=None):
    model.eval()

    for _ in range(max_tokens):
        idx_cond = idx if idx.size(1) <= model.config.block_size else idx[:, -model.config.block_size:]

        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature

        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[:, [-1]]] = -float('Inf')

        probs = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)

        idx = torch.cat((idx, idx_next), dim=1)

    return idx


def run_inference_pipeline(model, tokeniser, prompt, max_tokens=50, temperature=1.0, device='cpu'):
    input_ids = tokeniser.encode(prompt)  # list[int]
    idx = torch.tensor(input_ids, dtype=torch.long).unsqueeze(0)  # (1, seq_len)

    # device = next(model.parameters()).device
    idx = idx.to(device)
    model = model.to(device)
    output_ids = generate(model, idx, max_tokens, temperature, 50)

    output_tokens = output_ids[0].tolist()
    output_text = tokeniser.decode(output_tokens)

    return output_text

### create tokenizer

In [5]:
import tiktoken

tokenizer = tiktoken.get_encoding('gpt2')

# Generate some sample

In [ ]:
gpt_model = GPT.from_pretrained() # trained model
prompt = "Once upon a time in a small village "
run_inference_pipeline(gpt_model, tokenizer, prompt)

## Training and model evaluation

In [ ]:
def save_model(model):
    torch.save(model.state_dict(), "model.pth")
    print("model ins saved")

# evaluate_loss function
@torch.no_grad()
def evaluate_loss(model, data_loader, device, max_batches=None): # Added device
    model.eval()
    total_val_loss = 0.0
    batch_cnt = 0

    # Validation loss
    if len(data_loader) > 0:
        for i, (xb, yb) in enumerate(data_loader):
            if max_batches is not None and batch_cnt >= max_batches:
                break
            xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)
            _, loss = model(xb, yb)
            total_val_loss += loss.item()
            batch_cnt += 1
        avg_val_loss = total_val_loss / batch_cnt
    else:
        avg_val_loss = float('inf') # Handle empty val_loader

    model.train()
    return avg_val_loss

def train_one_epoch(model, train_loader, optimizer, scaler, config, device, epoch):
    model.train()
    epoch_loss = 0
    num_batches = 0

    for batch_idx, (xb, yb) in enumerate(train_loader):
        xb, yb = xb.to(device, non_blocking=True), yb.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if config.use_mixed_precision and scaler is not None:
            # Use autocast consistently
            try:
                # Use new API if available
                with torch.amp.autocast('cuda'):
                    logits, loss = model(xb, yb)
            except:
                # Fallback to old API
                with torch.cuda.amp.autocast():
                    logits, loss = model(xb, yb)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
            scaler.step(optimizer)
            scaler.update()
        else:
            logits, loss = model(xb, yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.grad_clip)
            optimizer.step()

        epoch_loss += loss.item()
        num_batches += 1

        # Print progress every 200 batches (less frequent)
        if batch_idx % 200 == 0:
            print(f"Epoch {epoch+1}/{config.num_epochs}, Batch {batch_idx}, Loss: {loss.item():.4f}")

    return epoch_loss / num_batches


def run_training_workflow(model, config, train_loader, val_loader, tokenizer, device, start_token="Once"):
    # Optimizer and scheduler
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config.learning_rate,
        weight_decay=config.weight_decay,
        betas=(0.9, 0.95)
    )

    scheduler = CosineAnnealingLR(optimizer, T_max=config.num_epochs)

    # Mixed precision training (updated API)
    if config.use_mixed_precision and torch.cuda.is_available():
        try:
            # Use new API if available (PyTorch 2.1+)
            scaler = torch.amp.GradScaler('cuda')
        except:
            # Fallback to old API
            scaler = torch.cuda.amp.GradScaler()
    else:
        scaler = None

    # Initialize lists to store metrics
    train_losses = []
    val_losses = []
    lrs = []

    # Early stopping parameters
    best_val_loss = float('inf')
    patience_counter = 0
    early_stopping_patience = 3 # Stop if val loss doesn't improve for 3 epochs
    best_model_state = None

    # Training loop
    print("Starting training...")
    for epoch in range(config.num_epochs):
        # Train one epoch
        avg_train_loss = train_one_epoch(model, train_loader, optimizer, scaler, config, device, epoch)

        # Update learning rate
        scheduler.step()

        # Validation
        val_loss = evaluate_loss(model, val_loader, device)

        # Store metrics
        train_losses.append(avg_train_loss)
        val_losses.append(val_loss)
        lrs.append(scheduler.get_last_lr()[0])

        print(f"Epoch {epoch+1}: Train Loss {avg_train_loss:.4f}, Val Loss {val_loss:.4f}, LR: {scheduler.get_last_lr()[0]:.6f}")

        # Early stopping check
        if config.use_early_stopping: # Only perform early stopping if enabled
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                patience_counter = 0
                best_model_state = model.state_dict() # Save best model state
                print(f"Validation loss improved. Best loss: {best_val_loss:.4f}")
            else:
                patience_counter += 1
                print(f"Validation loss did not improve. Patience: {patience_counter}/{early_stopping_patience}")
                if patience_counter >= early_stopping_patience:
                    print(f"Early stopping triggered after {epoch+1} epochs.")
                    break # Exit training loop

        # Generate sample text (less frequent)
        if epoch % 3 == 0 or epoch == config.num_epochs - 1:
            print("Generating sample text...")
            generated_text = run_inference_pipeline(model, tokenizer, start_token)
            print(f"Sample: {generated_text}...") # Truncate long samples
            print("-" * 50)

    print("Training completed!")

    # Load the best model state if early stopping occurred or if it's the end of training
    if config.use_early_stopping and best_model_state is not None:
        model.load_state_dict(best_model_state)
        print("Loaded best model state based on validation loss.")
    elif not config.use_early_stopping:
        print("Early stopping was disabled. Using model state from the last epoch.")

    # Save the trained model, config, and vocabulary mappings
    print("\nSaving model and configurations...")
    save_model(model)

    # Final generation
    model.eval()
    generated = run_inference_pipeline(model, tokenizer, start_token)
    print("\nFinal generated text:")
    print(generate)

    print("Done!")

    return train_losses, val_losses, lrs

## Visualization function

In [7]:
import matplotlib.pyplot as plt

def plot_training_metrics(train_losses, val_losses, lrs=None):
    """
    Plots the training loss, validation loss, and learning rate over epochs.

    Args:
        train_losses (list): List of training loss values per epoch.
        val_losses (list): List of validation loss values per epoch.
        lrs (list): List of learning rates per epoch.
    """

    epochs = range(1, len(train_losses) + 1)

    fig, axes = plt.subplots(2, 1, figsize=(10, 8))

    # Plot Training and Validation Loss
    axes[0].plot(epochs, train_losses, label='Training Loss', color='blue')
    axes[0].plot(epochs, val_losses, label='Validation Loss', color='red')
    axes[0].set_title('Training and Validation Loss Over Epochs')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True)

    # Plot Learning Rate Schedule
    if lrs is not None:
        axes[1].plot(epochs, lrs, label='Learning Rate', color='green')
        axes[1].set_title('Learning Rate Schedule Over Epochs')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Learning Rate')
        axes[1].legend()
        axes[1].grid(True)

    plt.tight_layout()
    plt.show()

print("Defined plot_training_metrics function.")

Defined plot_training_metrics function.


### model and training configs

In [5]:
@dataclass
class GPTConfig:
    # Model parameters
    block_size: int = 1024  # This is window size or context length
    vocab_size: int = 50304  # total number of tokens in tokeniser
    n_embd: int = 384 # 384 768      # Embedding dimmension size i.e each token is representation of this size
    n_layer: int = 2
    n_head: int = 12
    bias: bool = True
    embd_pdrop: float = 0.1
    dropout: float = 0.1
    attn_pdrop: float = 0.1
    resid_pdrop: float = 0.1
    attention_type: str = 'manual' # 'flash-attention', 'without-flash-attention', 'manual'

@dataclass
class TrainingConfig:
    # training parameters
    batch_size: int = 2
    num_epochs: int = 1
    learning_rate: float = 3e-4
    weight_decay: float = 0.05  # Increased for stronger regularization
    grad_clip: float = 1.0
    use_early_stopping: bool = True # New flag to control early stopping

    # Optimization
    compile_model: bool = True
    use_mixed_precision: bool = True

# FOR WINDOWS without triton
# torch.backends.cuda.enable_flash_sdp(False)
# torch.backends.cuda.enable_mem_efficient_sdp(False)
# torch.backends.cuda.enable_math_sdp(True)

# Prepare Dataset for Training

### local dataset 

In [10]:
from datasets import load_dataset

# reading the file
ds = load_dataset(
        "parquet",
        data_files="E:\code\DL\LLM-Lab\data\small_tiny_stories\part-00000-221bb1a7-7aa9-4850-bf6f-a9fdb712fa71-c000.snappy.parquet",
    )

train_test_split = 0.9
n = int(ds['train'].num_rows * train_test_split)

train_text = " ".join(ds['train'][:n]['text'])
val_text = " ".join(ds['train'][n:]['text'])

c:\Users\Devendra\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### online dataset

In [9]:
# text is a big string
import os
import random

train_path = os.path.join(path, "TinyStoriesV3-GPT4-train.txt")
val_path = os.path.join(path, "TinyStoriesV3-GPT4-valid.txt")

# Read all lines
with open(train_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

# Take 20% of lines randomly
sample_size = int(len(lines) * 0.1)
sample_lines = random.sample(lines, sample_size)

train_text = "".join(sample_lines)
print(f"train text : {len(train_text)}")

with open(val_path, "r", encoding="utf-8") as f:
    val_text = f.read()

print(f"val text : {len(val_text)}")

NameError: name 'path' is not defined

In [ ]:
train_dataset = TextDataset(train_text, tokenizer, GPTConfig.block_size)
val_datset = TextDataset(val_text, tokenizer, GPTConfig.block_size)

In [14]:
train_loader = get_data_loader(train_dataset, TrainingConfig.batch_size)
val_loader = get_data_loader(val_datset, TrainingConfig.batch_size)

print(f"train loader : {len(train_loader)}")
print(f"val loader : {len(val_loader)}")

train loader : 53328890
val loader : 5904291


# Training

In [ ]:
import os
import json # Required for config loading if we decide to load full config

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Initialize model based on the current (potentially modified) config and vocab_size
model = GPT(GPTConfig).to(device)

# Compile model for faster execution (PyTorch 2.0+) - Moved here to ensure model is compiled before loading state_dict
if TrainingConfig.compile_model and hasattr(torch, 'compile'):
    try:
        model = torch.compile(model)
        print("Model compiled successfully!")
    except Exception as e:
        print(f"Model compilation failed: {e}")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Run the training workflow and get the metrics
train_losses, val_losses, lrs = run_training_workflow(model, TrainingConfig(), train_loader, val_loader, tokenizer, device)

Using device: cuda
Total parameters: 23,259,648
Trainable parameters: 23,259,648
Starting training...
Epoch 1/1, Batch 0, Loss: 249.4222
Epoch 1/1, Batch 200, Loss: 15.1633
Epoch 1/1, Batch 400, Loss: 7.6750
Epoch 1/1, Batch 600, Loss: 6.4094
Epoch 1/1, Batch 800, Loss: 5.9240
Epoch 1/1, Batch 1000, Loss: 5.0010
Epoch 1/1, Batch 1200, Loss: 5.2099
Epoch 1/1, Batch 1400, Loss: 4.7223
Epoch 1/1, Batch 1600, Loss: 5.2486
Epoch 1/1, Batch 1800, Loss: 4.6773
Epoch 1/1, Batch 2000, Loss: 4.7162
Epoch 1/1, Batch 2200, Loss: 4.6422
Epoch 1/1, Batch 2400, Loss: 3.9761


KeyboardInterrupt: 

In [16]:
# Clear GPU memory
import gc
import torch

# Safely delete model, optimizer, and scheduler if they exist
if 'model' in locals() and model is not None:
    del model
if 'optimizer' in locals() and optimizer is not None:
    del optimizer
if 'scheduler' in locals() and scheduler is not None:
    del scheduler

gc.collect()
torch.cuda.empty_cache()
print("GPU memory cleared.")

GPU memory cleared.


In [1]:
import torch

def estimate_model_gpu_memory(model, batch_size=1, seq_len=None, dtype=torch.float32, device=None):
    """
    Estimate GPU memory required to run a model.
    
    Args:
        model: nn.Module
        batch_size: batch size for dummy forward pass
        seq_len: sequence length (if applicable)
        dtype: torch data type (float32, float16)
        device: torch device (default: cuda:0)
    """
    if device is None:
        device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    
    model = model.to(device)
    model.eval()
    
    # Model parameters
    num_params = sum(p.numel() for p in model.parameters())
    bytes_per_param = torch.tensor([], dtype=dtype).element_size()
    param_mem = num_params * bytes_per_param  # bytes
    
    # Optimizer memory (roughly 2x parameters for Adam)
    optimizer_mem = param_mem * 2
    
    # Prepare a dummy input to estimate activation memory
    # Assumes first layer input shape is (seq_len, embedding_dim) or similar
    sample = None
    for module in model.modules():
        if hasattr(module, 'weight') and module.weight is not None:
            sample_shape = module.weight.shape
            break
    
    # Create dummy batch
    dummy_input = torch.randn(batch_size, *(sample_shape[1:]), device=device, dtype=dtype)
    
    try:
        torch.cuda.reset_peak_memory_stats(device)
        with torch.no_grad():
            model(dummy_input)
        peak_mem = torch.cuda.max_memory_allocated(device)  # bytes
    except:
        peak_mem = 0
        print("Warning: Could not compute peak memory from forward pass.")
    
    total_mem = param_mem + optimizer_mem + peak_mem
    
    print(f"Model parameters memory: {param_mem / (1024**2):.2f} MB")
    print(f"Estimated optimizer memory: {optimizer_mem / (1024**2):.2f} MB")
    print(f"Estimated activation memory (forward pass): {peak_mem / (1024**2):.2f} MB")
    print(f"Total estimated GPU memory: {total_mem / (1024**3):.2f} GB")
    
    return total_mem

In [12]:
import sys
import torch

def estimate_token_memory(tokens, dtype=torch.long, batch_size=None):
    """
    Estimate memory usage for a list of tokens.

    Args:
        tokens: list of list of integers (tokenized dataset)
        dtype: torch data type for token tensor
        batch_size: optional, estimate memory per batch
    """
    # --- 1. Python object memory ---
    py_mem = sys.getsizeof(tokens)
    for t in tokens:
        py_mem += sys.getsizeof(t)
        for token in t:
            py_mem += sys.getsizeof(token)

    print(f"Python objects memory: {py_mem / (1024**2):.2f} MB ({py_mem / (1024**3):.3f} GB)")

    # --- 2. Token tensor memory ---
    total_tokens = sum(len(t) for t in tokens)
    tensor_mem = total_tokens * torch.tensor([], dtype=dtype).element_size()
    print(f"Estimated memory for token tensor: {tensor_mem / (1024**2):.2f} MB ({tensor_mem / (1024**3):.3f} GB)")

    # --- 3. Optional batch memory ---
    if batch_size:
        max_seq_len_batch = max(len(t) for t in tokens[:batch_size])
        batch_mem = batch_size * max_seq_len_batch * torch.tensor([], dtype=dtype).element_size()
        print(f"Estimated memory per batch (batch_size={batch_size}): {batch_mem / (1024**2):.2f} MB")
    else:
        batch_mem = None

    return {
        "python_mem_bytes": py_mem,
        "num_tokens": total_tokens,
        "token_tensor_mem_bytes": tensor_mem,
        "batch_mem_bytes": batch_mem
    }

In [23]:
import torch

def gpu_memory_planner(seq_len, dtype=torch.float16, safety_factor=0.8):
    """
    Get GPU capabilities and estimate how much data it can hold and process efficiently.

    Args:
        seq_len (int): number of tokens per sequence
        dtype: torch dtype of the input (float16 or float32)
        safety_factor (float): fraction of free memory to use (to avoid OOM)

    Returns:
        list of dict: summary for each GPU
    """
    if not torch.cuda.is_available():
        print("No GPU detected.")
        return None

    gpus = []
    num_gpus = torch.cuda.device_count()

    for i in range(num_gpus):
        torch.cuda.synchronize(i)
        props = torch.cuda.get_device_properties(i)
        free_mem, total_mem_cuda = torch.cuda.mem_get_info(i)
        free_mem_gb = free_mem / (1024**3)
        total_mem_gb = total_mem_cuda / (1024**3)

        # Bytes per token (for input tensor, typically float16/float32)
        bytes_per_token = torch.tensor([], dtype=dtype).element_size()

        # Estimate safe tokens/batch
        safe_mem_bytes = free_mem * safety_factor
        max_tokens = int(safe_mem_bytes / bytes_per_token)
        max_batch_size = max_tokens // seq_len

        gpu_info = {
            "gpu_id": i,
            "gpu_name": props.name,
            "total_mem_gb": total_mem_gb,
            "free_mem_gb": free_mem_gb,
            "cuda_cores": props.multi_processor_count,
            "compute_capability": f"{props.major}.{props.minor}",
            "max_batch_size_estimate": max_batch_size,
            "seq_len_per_batch": seq_len,
            "dtype": str(dtype)
        }

        print(f"GPU {i}: {props.name}")
        print(f"  Total Memory: {total_mem_gb:.2f} GB, Free Memory: {free_mem_gb:.2f} GB")
        print(f"  CUDA Cores: {props.multi_processor_count}, Compute Capability: {props.major}.{props.minor}")
        print(f"  Estimated max batch size for {seq_len} tokens/seq: {max_batch_size}\n")

        gpus.append(gpu_info)

    return gpus

In [13]:
model = GPT(GPTConfig())

estimate_model_gpu_memory(model, batch_size=4, seq_len=GPTConfig.block_size, dtype=torch.float32, device='cuda')

Model parameters memory: 88.73 MB
Estimated optimizer memory: 177.46 MB
Estimated activation memory (forward pass): 0.00 MB
Total estimated GPU memory: 0.26 GB


279115776

In [24]:
gpu_stats = gpu_memory_planner(seq_len=1024, dtype=torch.float16)

GPU 0: NVIDIA GeForce GTX 1650
  Total Memory: 4.00 GB, Free Memory: 2.58 GB
  CUDA Cores: 14, Compute Capability: 7.5
  Estimated max batch size for 1024 tokens/seq: 1083473



In [25]:
gpu_stats

[{'gpu_id': 0,
  'gpu_name': 'NVIDIA GeForce GTX 1650',
  'total_mem_gb': 3.999755859375,
  'free_mem_gb': 2.583203125745058,
  'cuda_cores': 14,
  'compute_capability': '7.5',
  'max_batch_size_estimate': 1083473,
  'seq_len_per_batch': 1024,
  'dtype': 'torch.float16'}]